# Merged BERTopic - Analysis

Purpose: load the finalized frozen merged BERTopic artefacts and rerun all inspections, exports, and visualizations without rebuilding the merged model.

Thesis note: this notebook prefers the rerun frozen thesis bundle `final_merged_topics_thesis_final_rerun` if it exists; otherwise it falls back to `final_merged_topics_thesis_final`.

## 1. Setup and load the frozen thesis-final bundle

This notebook loads the preferred frozen bundle directly instead of the mutable canonical merged-output paths.

In [ ]:
from pathlib import Path
import math
import re
import sys

import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from bertopic import BERTopic

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name != "Thesis":
    PROJECT_ROOT = PROJECT_ROOT.parent

MODULE_DIR = PROJECT_ROOT / "02_TopicModeling"
if str(MODULE_DIR) not in sys.path:
    sys.path.insert(0, str(MODULE_DIR))

import merged_outlets_analysis as moa

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 140)

FROZEN_RUNS_DIR = PROJECT_ROOT / "02_TopicModeling" / "outputs" / "frozen_merged_runs"
PREFERRED_RUN_LABELS = [
    "final_merged_topics_thesis_final_rerun",
    "final_merged_topics_thesis_final",
]
FINAL_BUNDLE_DIR = next(
    (FROZEN_RUNS_DIR / label for label in PREFERRED_RUN_LABELS if (FROZEN_RUNS_DIR / label).exists()),
    FROZEN_RUNS_DIR / PREFERRED_RUN_LABELS[-1],
)
MERGED_MODEL_DIR = FINAL_BUNDLE_DIR / "merged_model"
ARTICLES_PATH = FINAL_BUNDLE_DIR / "merged_articles_with_umap.csv"
TOPIC_INFO_PATH = FINAL_BUNDLE_DIR / "merged_topic_info_display.csv"
METADATA_PATH = FINAL_BUNDLE_DIR / "merged_analysis_metadata.json"
TOP30_WORDS_PATH = FINAL_BUNDLE_DIR / "merged_topic_info_top30_words.csv"
CONSISTENT_LABELS_PATH = FINAL_BUNDLE_DIR / "topics_by_count_top30_words_consistent_labels.csv"
CONSISTENT_LABELS_EXPORT_PATH = PROJECT_ROOT / "data" / "processed" / "topics_by_count_top30_words_consistent_labels.csv"
ARTICLE_TOPIC_LABEL_EXPORT_PATH = PROJECT_ROOT / "data" / "processed" / "df_combined_with_topic_label_current.csv"
BUNDLE_ARTICLE_TOPIC_LABEL_EXPORT_PATH = FINAL_BUNDLE_DIR / "df_combined_with_topic_label_current.csv"

merged_model = BERTopic.load(MERGED_MODEL_DIR, embedding_model=moa.EMBEDDING_MODEL_NAME)
merged_articles_base, merged_topic_info_base, merged_metadata = moa.load_merged_analysis_cache(
    PROJECT_ROOT,
    articles_path=ARTICLES_PATH,
    topic_info_path=TOPIC_INFO_PATH,
    metadata_path=METADATA_PATH,
)
merged_topic_info_top30 = pd.read_csv(TOP30_WORDS_PATH) if TOP30_WORDS_PATH.exists() else None
consistent_topic_labels = pd.read_csv(CONSISTENT_LABELS_PATH) if CONSISTENT_LABELS_PATH.exists() else None
clean_assignments = moa.build_traceable_merged_assignment_view(PROJECT_ROOT, merged_articles_base)
plot_articles_viz = moa.ensure_outlet_key_columns(merged_articles_base.copy())

print("Loaded final thesis bundle from:", FINAL_BUNDLE_DIR)
print("Loaded merged model from:", MERGED_MODEL_DIR)
if merged_topic_info_top30 is not None:
    print("Loaded top-30 words from:", TOP30_WORDS_PATH)
if consistent_topic_labels is not None:
    print("Loaded consistent topic labels from:", CONSISTENT_LABELS_PATH)
print(merged_metadata)


## 2. Topic overview

In [ ]:
clean_topics = (
    merged_topic_info_base.loc[
        merged_topic_info_base["Topic"] != -1,
        ["DisplayTopic", "Topic", "DisplayLabel", "Name", "Count", "Representation"],
    ]
    .rename(columns={
        "DisplayTopic": "topic_nr",
        "Topic": "topic_id",
        "DisplayLabel": "topic_label",
        "Name": "topic_name_raw",
        "Count": "article_count",
    })
    .sort_values(["topic_nr", "topic_id"])
    .reset_index(drop=True)
)

if merged_topic_info_top30 is not None and {"Topic", "TopWords30Str"}.issubset(merged_topic_info_top30.columns):
    top30_map = dict(zip(merged_topic_info_top30["Topic"], merged_topic_info_top30["TopWords30Str"]))
    clean_topics["top_30_words"] = clean_topics["topic_id"].map(top30_map).fillna("")
else:
    clean_topics["top_30_words"] = clean_topics["topic_id"].apply(
        lambda topic_id: ", ".join(word for word, _ in (merged_model.get_topic(int(topic_id)) or [])[:30])
    )

if consistent_topic_labels is None and merged_topic_info_top30 is not None and {"DisplayTopic", "Topic", "DisplayLabel", "Count", "TopWords30Str"}.issubset(merged_topic_info_top30.columns):
    consistent_topic_labels = moa.build_consistent_topic_label_table(
        merged_topic_info_top30,
        top_words_col="TopWords30Str",
    )

if consistent_topic_labels is not None:
    label_col = "topic_label" if "topic_label" in consistent_topic_labels.columns else "topic_label_consistent"
    consistent_label_map = dict(zip(consistent_topic_labels["topic_id"], consistent_topic_labels[label_col]))
    clean_topics["topic_label"] = clean_topics["topic_id"].map(consistent_label_map).fillna(clean_topics["topic_label"])

consistent_topic_labels = (
    clean_topics.loc[:, ["topic_nr", "topic_id", "topic_label", "article_count", "top_30_words"]]
    .sort_values(["article_count", "topic_nr"], ascending=[False, True])
    .reset_index(drop=True)
)
consistent_topic_labels.to_csv(CONSISTENT_LABELS_PATH, index=False)
consistent_topic_labels.to_csv(CONSISTENT_LABELS_EXPORT_PATH, index=False)

display(clean_topics.head(30))
print("Substantive merged topics:", len(clean_topics))
print("Saved consistent topic table to:", CONSISTENT_LABELS_PATH)
print("Saved processed copy to:", CONSISTENT_LABELS_EXPORT_PATH)


## 3. Topic label scaffold (edit `custom_label` and rerun the next cell)

In [ ]:
label_map_path = PROJECT_ROOT / "data" / "processed" / "topic_label_map.csv"

def auto_short_label(text: str) -> str:
    text = re.sub(r"^Topic\s+\d+\s+[—-]\s*", "", str(text))
    text = re.sub(r"^\d+_", "", text)
    text = text.replace("_", " ")
    text = re.sub(r"\s+", " ", text).strip()
    return text

topic_label_scaffold = clean_topics[["topic_nr", "topic_id", "topic_label", "article_count"]].copy()
topic_label_scaffold["current_label"] = topic_label_scaffold["topic_label"].map(auto_short_label)

if label_map_path.exists():
    existing = pd.read_csv(label_map_path)
    if {"topic_id", "custom_label"}.issubset(existing.columns):
        topic_label_scaffold = topic_label_scaffold.merge(
            existing[["topic_id", "custom_label"]],
            on="topic_id",
            how="left",
        )
    else:
        topic_label_scaffold["custom_label"] = topic_label_scaffold["current_label"]
else:
    topic_label_scaffold["custom_label"] = topic_label_scaffold["current_label"]

topic_label_scaffold["custom_label"] = (
    topic_label_scaffold["custom_label"]
    .fillna(topic_label_scaffold["current_label"])
    .astype(str)
    .str.strip()
)

topic_label_scaffold = topic_label_scaffold[["topic_nr", "topic_id", "article_count", "current_label", "custom_label"]]
topic_label_scaffold.to_csv(label_map_path, index=False)

display(topic_label_scaffold.head(20))
print("Saved label scaffold to:", label_map_path)


In [ ]:
label_df = pd.read_csv(label_map_path)
label_df["custom_label"] = label_df["custom_label"].fillna("").astype(str).str.strip()
custom_label_map = dict(
    zip(
        label_df.loc[label_df["custom_label"].ne(""), "topic_id"].astype(int),
        label_df.loc[label_df["custom_label"].ne(""), "custom_label"],
    )
)

merged_topic_info_viz = merged_topic_info_base.copy()
merged_topic_info_viz = moa.apply_topic_name_overrides(merged_topic_info_viz, custom_label_map, label_col="CustomLabel")
merged_topic_info_viz["TopicNameClean"] = merged_topic_info_viz["CustomLabel"]
merged_topic_info_viz["DisplayLabel"] = merged_topic_info_viz.apply(
    lambda row: "Outliers" if int(row["Topic"]) == -1 else f"Topic {int(row['DisplayTopic'])} — {row['CustomLabel']}",
    axis=1,
)

merged_model_viz = merged_model
merged_model_viz.set_topic_labels(custom_label_map)

display(merged_topic_info_viz.loc[merged_topic_info_viz["Topic"] != -1, ["DisplayTopic", "Topic", "DisplayLabel", "Count"]].head(20))


## 4. Outlet x topic tables

In [ ]:
outlet_order = [spec.label for spec in moa.OUTLET_SPECS.values()]

topic_outlet_pivot = (
    clean_assignments.loc[clean_assignments["topic_id"].notna()]
    .groupby(["topic_nr", "topic_id", "topic_label", "outlet"])
    .size()
    .unstack(fill_value=0)
    .reindex(columns=outlet_order, fill_value=0)
    .reset_index()
    .sort_values(["topic_nr", "topic_id"])
    .reset_index(drop=True)
)

topic_outlet_pivot["total_articles"] = topic_outlet_pivot[outlet_order].sum(axis=1)
topic_outlet_pivot["dominant_outlet"] = topic_outlet_pivot[outlet_order].idxmax(axis=1)
topic_outlet_pivot["dominant_count"] = topic_outlet_pivot[outlet_order].max(axis=1)
topic_outlet_pivot["dominant_share"] = (
    topic_outlet_pivot["dominant_count"] / topic_outlet_pivot["total_articles"]
).round(3)

pivot_path = PROJECT_ROOT / "data" / "processed" / "topic_outlet_pivot.csv"
pivot_sorted_path = PROJECT_ROOT / "data" / "processed" / "topic_outlet_pivot_sorted.csv"
topic_outlet_pivot.to_csv(pivot_path, index=False)
topic_outlet_pivot.sort_values(["dominant_share", "total_articles"], ascending=[False, False]).to_csv(pivot_sorted_path, index=False)

display(topic_outlet_pivot.head(20))
display(topic_outlet_pivot.sort_values(["dominant_share", "total_articles"], ascending=[False, False]).head(20))
print("Saved to:", pivot_path)
print("Saved sorted view to:", pivot_sorted_path)


## 5. Article-level exports

In [ ]:
topic_label_map = dict(zip(consistent_topic_labels["topic_id"], consistent_topic_labels["topic_label"]))
topic_id_col = "topic_id" if "topic_id" in clean_assignments.columns else "merged_topic"

df_combined_with_topic_label_current = (
    clean_assignments.loc[:, ["Date", "Title", "Text", "outlet", "row_id", topic_id_col]]
    .rename(columns={"outlet": "source", topic_id_col: "topic_id"})
    .reset_index(drop=True)
)
df_combined_with_topic_label_current["topic_label_current"] = (
    df_combined_with_topic_label_current["topic_id"].map(topic_label_map).fillna("Outliers")
)
df_combined_with_topic_label_current = df_combined_with_topic_label_current[
    ["Date", "Title", "Text", "source", "row_id", "topic_label_current"]
]

df_combined_with_topic_label_current.to_csv(ARTICLE_TOPIC_LABEL_EXPORT_PATH, index=False)
df_combined_with_topic_label_current.to_csv(BUNDLE_ARTICLE_TOPIC_LABEL_EXPORT_PATH, index=False)

SAMPLE_N = 500
RANDOM_STATE = 42
sampled_parts = [
    g.sample(n=min(SAMPLE_N, len(g)), random_state=RANDOM_STATE)
    for _, g in df_combined_with_topic_label_current.groupby("source", sort=True)
]
sampled_500_per_source = (
    pd.concat(sampled_parts, ignore_index=True)
    .sort_values(["source", "Date", "row_id"])
    .reset_index(drop=True)
)

sample_path = PROJECT_ROOT / "data" / "processed" / "df_combined_with_topic_label_current_sample_500_per_source.csv"
sampled_500_per_source.to_csv(sample_path, index=False)

display(df_combined_with_topic_label_current.head(10))
display(sampled_500_per_source["source"].value_counts())
print("Saved full article export to:", ARTICLE_TOPIC_LABEL_EXPORT_PATH)
print("Saved bundle copy to:", BUNDLE_ARTICLE_TOPIC_LABEL_EXPORT_PATH)
print("Saved balanced source sample to:", sample_path)


## 6. Save BERTopic interactive visuals as HTML

In [ ]:
visual_dir = PROJECT_ROOT / "02_TopicModeling" / "outputs" / "merged_visuals_pretty"
visual_dir.mkdir(parents=True, exist_ok=True)

merged_model_viz.visualize_topics(
    top_n_topics=len(clean_topics),
    custom_labels=True,
    width=1200,
    height=900,
    title="<b>Merged BERTopic - Intertopic Distance Map</b>",
).write_html(visual_dir / "intertopic_distance_map_pretty.html")

merged_model_viz.visualize_barchart(
    top_n_topics=len(clean_topics),
    n_words=10,
    custom_labels=True,
    width=320,
    height=260,
).write_html(visual_dir / "topic_barchart_pretty.html")

merged_model_viz.visualize_heatmap(
    top_n_topics=len(clean_topics),
    custom_labels=True,
    width=1100,
    height=900,
).write_html(visual_dir / "topic_heatmap_pretty.html")

merged_model_viz.visualize_hierarchy(
    top_n_topics=len(clean_topics),
    custom_labels=True,
    width=1400,
    height=1800,
).write_html(visual_dir / "topic_hierarchy_pretty.html")

print("Saved BERTopic HTML visuals to:", visual_dir)


## 7. Balanced outlet UMAP

In [ ]:
balanced_n = int(moa.ensure_outlet_key_columns(plot_articles_viz)["outlet_key"].value_counts().min())

balanced_articles_viz = (
    moa.ensure_outlet_key_columns(plot_articles_viz)
    .groupby("outlet_key", group_keys=False)
    .apply(lambda g: g.sample(n=balanced_n, random_state=42))
    .reset_index(drop=True)
)

fig, ax = moa.plot_outlet_colored_topic_umap(
    balanced_articles_viz,
    merged_topic_info_viz,
    top_n=12,
    figsize=(18, 12),
    point_size=10,
    alpha=0.55,
    include_raw_topic_id_in_labels=False,
)
ax.set_title(f"Balanced Outlet UMAP - {balanced_n} Articles per Outlet")
plt.show()


## 8. Normalized outlet dominance map (class-imbalance corrected)

In [ ]:
def plot_normalized_outlet_dominance_map(
    articles_df,
    topic_info_df,
    *,
    bins=75,
    top_n_labels=12,
    background_sample=5000,
    min_cell_mass_quantile=0.50,
    figsize=(18, 12),
    random_state=42,
):
    df = moa.ensure_outlet_key_columns(articles_df)
    df = df.loc[df["merged_topic"] != -1].copy()

    outlet_keys = list(moa.OUTLET_SPECS.keys())
    outlet_sizes = df["outlet_key"].value_counts()
    df["outlet_weight"] = df["outlet_key"].map(lambda k: 1.0 / outlet_sizes[k])

    x_edges = np.linspace(df["umap_x"].min(), df["umap_x"].max(), bins + 1)
    y_edges = np.linspace(df["umap_y"].min(), df["umap_y"].max(), bins + 1)

    df["xbin"] = np.clip(np.digitize(df["umap_x"], x_edges) - 1, 0, bins - 1)
    df["ybin"] = np.clip(np.digitize(df["umap_y"], y_edges) - 1, 0, bins - 1)

    cell = (
        df.groupby(["xbin", "ybin", "outlet_key"])["outlet_weight"]
        .sum()
        .unstack(fill_value=0)
        .reindex(columns=outlet_keys, fill_value=0)
    )

    cell["total_weight"] = cell.sum(axis=1)
    cell = cell[cell["total_weight"] > 0].copy()

    mass_cutoff = cell["total_weight"].quantile(min_cell_mass_quantile)
    cell = cell[cell["total_weight"] >= mass_cutoff].copy()

    cell["dominant_outlet_key"] = cell[outlet_keys].idxmax(axis=1)
    cell["dominant_share"] = cell[outlet_keys].max(axis=1) / cell["total_weight"]

    baseline = 1 / len(outlet_keys)
    cell["certainty"] = ((cell["dominant_share"] - baseline) / (1 - baseline)).clip(0, 1)

    x_centers = (x_edges[:-1] + x_edges[1:]) / 2
    y_centers = (y_edges[:-1] + y_edges[1:]) / 2

    cell = cell.reset_index()
    cell["x"] = cell["xbin"].map(dict(enumerate(x_centers)))
    cell["y"] = cell["ybin"].map(dict(enumerate(y_centers)))

    mass_scale = (cell["total_weight"] / cell["total_weight"].quantile(0.95)).clip(0.2, 1.0)
    cell["size"] = 35 + 160 * np.sqrt(mass_scale)

    fig, ax = plt.subplots(figsize=figsize)

    bg = df.sample(n=min(background_sample, len(df)), random_state=random_state)
    ax.scatter(bg["umap_x"], bg["umap_y"], c="#E8E8E8", s=3, alpha=0.10, linewidths=0, rasterized=True, zorder=1)

    for outlet_key, spec in moa.OUTLET_SPECS.items():
        sub = cell.loc[cell["dominant_outlet_key"] == outlet_key].copy()
        if sub.empty:
            continue
        rgba = [mcolors.to_rgba(moa.OUTLET_COLOR_MAP[outlet_key], alpha=0.15 + 0.75 * a) for a in sub["certainty"]]
        ax.scatter(sub["x"], sub["y"], s=sub["size"], c=rgba, marker="s", linewidths=0, label=spec.label, zorder=3)

    top_topic_ids = (
        topic_info_df.loc[topic_info_df["Topic"] != -1]
        .nlargest(top_n_labels, "Count")["Topic"]
        .tolist()
    )

    centroids = (
        df.loc[df["merged_topic"].isin(top_topic_ids)]
        .groupby("merged_topic", as_index=False)[["umap_x", "umap_y"]]
        .mean()
    )

    label_map = dict(zip(topic_info_df["Topic"], topic_info_df["DisplayLabel"]))
    for _, row in centroids.iterrows():
        topic_id = int(row["merged_topic"])
        label = str(label_map.get(topic_id, ""))
        if "—" in label:
            label = label.split("—", 1)[1].strip()
        ax.text(
            row["umap_x"],
            row["umap_y"],
            label,
            fontsize=9,
            ha="center",
            va="center",
            bbox={"facecolor": "white", "alpha": 0.90, "edgecolor": "none", "pad": 1.2},
            zorder=5,
        )

    ax.set_title("Normalized Outlet Dominance Map - Class-Imbalance Corrected")
    ax.set_xticks([])
    ax.set_yticks([])
    ax.legend(title="Dominant outlet", frameon=False, loc="upper right")
    plt.show()

plot_normalized_outlet_dominance_map(plot_articles_viz, merged_topic_info_viz, bins=75, top_n_labels=12)


## 9. Outlet-vs-rest highlight maps

In [ ]:
for outlet_key in moa.OUTLET_SPECS.keys():
    fig, ax = moa.plot_outlet_highlight_umap(
        plot_articles_viz,
        outlet_key=outlet_key,
        merged_topic_info=merged_topic_info_viz,
        figsize=(14, 10),
    )
    plt.show()


## 10. Optional 3D UMAP

In [ ]:
prepared_by_outlet = moa.load_all_prepared_documents(PROJECT_ROOT)
combined_prepared = moa.combine_prepared_documents(prepared_by_outlet)
merged_articles_3d, merged_topic_info_3d = moa.build_merged_article_umap_3d(
    merged_model,
    combined_prepared,
    umap_n_neighbors=10,
    umap_min_dist=0.0,
    random_state=42,
)

fig, ax = moa.plot_merged_topic_umap_3d(
    merged_articles_3d,
    merged_topic_info_3d,
    top_n=20,
    figsize=(14, 10),
)
plt.show()
